In [ ]:
!jq '. | length' ../data/dataset.json

In [ ]:

!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

In [ ]:
from semantic_engine_demo.json_load import load_json
from pathlib import Path

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"

data_path = DATA_DIR / "dataset.json"

In [ ]:
import numpy as np

embs_np = np.load(DATA_DIR / "embeddings_02.npy").astype(np.float64)

embs_np[:5]

In [ ]:
from arrowspace import ArrowSpaceBuilder
import time

graph_params = {
    "eps": 0.8,
    "k": 25,
    "topk": 20,
    "p": 2.0,
    "sigma": None,
}

print(f"Graph parameters: {graph_params}")

print("Building ArrowSpace...")
start = time.perf_counter()
aspace, gl = (
    ArrowSpaceBuilder()
    .with_seed(42)
    .with_dims_reduction(enabled=False, eps=None)
    .with_sampling("simple", 1.0)
).build_and_store(graph_params, embs_np)
print(f"Build time: {time.perf_counter() - start:.2f}s")

In [ ]:
corpus = load_json(data_path)

In [ ]:
import sqlite3

conn = sqlite3.connect('local_data.db')
cursor = conn.cursor()

In [ ]:


def create_table(crs):
    
    # 2. Define and execute the table creation query
    # We'll create a sample 'users' table. 
    # IF NOT EXISTS prevents errors if you run this cell multiple times.
    create_table_query = '''
    CREATE TABLE IF NOT EXISTS docs_idx (
        id INTEGER PRIMARY KEY ,
        author_reputation INTEGER NOT NULL,
        version INTEGER NOT NULL,
        fork_count INTEGER NOT NULL,
        likes INTEGER NOT NULL,
        upvotes INTEGER NOT NULL,
        downvotes INTEGER NOT NULL,
        views INTEGER NOT NULL,
        uses INTEGER NOT NULL

    )
    '''
    crs.execute(create_table_query)

    # Commit the changes to the database
    

  

In [ ]:
create_table(cursor)
conn.commit()
print("Database connected and 'users' table initialized.")

In [ ]:

insert_query = '''
INSERT INTO docs_idx (id, author_reputation, version, fork_count, likes, upvotes, downvotes, views, uses)
VALUES (:pos , :author_reputation, :version, :fork_count, :likes, :upvotes, :downvotes, :views, :uses)
'''

try:
    cursor.executemany(insert_query, corpus)

# 4. Commit to save the records to disk
    conn.commit()
except:
    pass

print(f"Successfully inserted {cursor.rowcount} records into the database.")

In [ ]:
import pandas as pd

# Query the database and load it directly into a DataFrame
df = pd.read_sql_query("SELECT * FROM docs_idx WHERE id = 0", conn)

# Display the formatted table
display(df)

# Note: It's good practice to close the connection when you're completely finished
conn.close()

In [ ]:
import numpy as np

queries_path = DATA_DIR / "benchmark" / "queries_emb.npy"

queries_emb = np.load(queries_path)

In [ ]:
result = aspace.search(queries_emb[2], gl, 0.8)

result

In [ ]:
queries_path_1 = DATA_DIR / "benchmark" / "benchmark_queries.json"
queries_corpus = load_json(queries_path_1)

In [ ]:
queries_corpus[2]

In [ ]:
corpus[13080]

In [ ]:
ids = [r[0] for r in result]

# 1. Fetch data safely
placeholders = ",".join(["?"] * len(ids))

case_clauses = " ".join([f"WHEN ? THEN {i}" for i in range(len(ids))])
order_query = f"ORDER BY CASE id {case_clauses} END"


query = f"SELECT * FROM docs_idx WHERE id IN ({placeholders}) {order_query}"

with sqlite3.connect('local_data.db') as conn:
    df = pd.read_sql_query(query, conn, params=ids + ids)

display(df)

In [ ]:
from semantic_engine_demo.benchmark_evaluation import extract_and_evaluate


predicted_results = result 
ground_truth_item = queries_corpus[2] 

# Run the evaluation
single_eval = extract_and_evaluate(
    predicted_results=predicted_results, 
    query_corpus_item=ground_truth_item, 
    k=10
)

print("Single Engine Evaluation:")
for key, value in single_eval.items():
    print(f"{key}: {value}")